# The A Way Student-Facing Agent

Complete, executable notebook for the C3 Introduction to Agents assignment. It defines the business identity, separate persona and instructions, a three-way router, two MCP-style tools, an OpenAI tool-calling loop, bounded session memory, required demonstrations, and a Gradio interface. The demonstrations use a local scripted provider so the notebook runs safely without an API key; the application uses OpenAI when a key is configured.

## 1. Setup and business sources

In [ ]:
import json
import logging
import os
import tempfile
from pathlib import Path
from types import SimpleNamespace

from src.config import Settings, load_settings

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)
ORIGINAL_DATA_DIR_ENV = os.environ.get("THE_A_WAY_DATA_DIR")
settings = load_settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)
print(f"Configured model: {settings.openai_model}")
print(f"API key configured: {settings.has_api_key}")
print(f"Runtime data directory: {settings.data_dir.resolve()}")

In [ ]:
from src.business_sources import format_source_report, load_business_sources
from src.llm_client import load_business_knowledge

sources = load_business_sources(
    Path("me/about_business.pdf"),
    Path("me/business_summary.txt"),
    Path("data/course_catalog.json"),
)
assert not sources.conflicts, sources.conflicts
BUSINESS_KNOWLEDGE = load_business_knowledge()
print(format_source_report(sources))

## 2. Persona and instructions

These are deliberately separate so identity/tone remains distinct from operational guardrails.

In [ ]:
from src.identity import PERSONA

# PERSONA: who the agent is and how it speaks.
print("PERSONA\n-------")
print(PERSONA)

In [ ]:
from src.identity import INSTRUCTIONS, build_system_prompt

# INSTRUCTIONS: the agent's operational rules and safety boundaries.
print("INSTRUCTIONS\n------------")
print(INSTRUCTIONS)
SYSTEM_PROMPT = build_system_prompt()

## 3. Three-way intent router

The deterministic router chooses exactly one path. Lead collection wins for mixed lead/feedback requests so an in-progress contact flow is not lost.

In [ ]:
from src.intent import classify_intent

router_examples = [
    "What services does The A Way provide?",
    "I want to join EECE 210.",
    "I have a complaint about tutoring.",
]
for example in router_examples:
    print(f"{classify_intent(example, []):<18} <- {example}")

## 4. MCP-style tool contracts

`record_customer_interest(email, name, message)` and `record_feedback(question)` validate their inputs, log calls, save idempotent local SQLite records, and return structured results. Interest logging never implies registration; feedback logging never implies resolution.

In [ ]:
from src.tools import TOOL_SCHEMAS, record_customer_interest, record_feedback

print(json.dumps(TOOL_SCHEMAS, indent=2))
print("Callable tools:", record_customer_interest.__name__, record_feedback.__name__)

## 5. OpenAI agent and short-term memory

The real adapter sends the system prompt, extracted PDF text, summary, catalog, bounded history, and both tool schemas to OpenAI. It validates and executes tool calls, returns tool results to the model, and handles provider failures safely.

In [ ]:
from src.assistant import StudentConcierge
from src.llm_client import LLMResult, OpenAIStudentConcierge

live_agent = OpenAIStudentConcierge(settings=settings)
print("OpenAI adapter ready. A network call is made only after chat() and requires OPENAI_API_KEY.")

## 6. Required demonstrations (offline and reproducible)

A scripted model stands in for OpenAI for deterministic classroom demonstration. Production uses the same `StudentConcierge` orchestration with the live adapter. Tool persistence is redirected to a temporary directory.

In [ ]:
class ScriptedDemoModel:
    def __init__(self):
        self.calls = []

    def chat(self, **request):
        self.calls.append(request)
        message = request["user_message"].casefold()
        if "what did i just ask" in message:
            prior = request["history"][-2]["content"]
            text = f"You previously asked: {prior}"
        elif "eece 210" in message:
            text = ("EECE 210 is in the catalog, but no active offering, "
                    "schedule, tutor, price, or open registration is confirmed.")
        elif "register" in message or "payment" in message:
            text = ("I cannot register students or process payments. I can only "
                    "record interest for academy follow-up.")
        else:
            text = ("The A Way is a fictional university tutoring academy offering "
                    "course-focused support, study planning, and exam preparation.")
        return LLMResult(success=True, text=text)

demo_storage = tempfile.TemporaryDirectory(prefix="the_a_way_demo_")
os.environ["THE_A_WAY_DATA_DIR"] = demo_storage.name
demo_model = ScriptedDemoModel()
demo_agent = StudentConcierge(client=demo_model, business_knowledge=BUSINESS_KNOWLEDGE)

In [ ]:
print("SCENARIO 1 - General business question")
print(demo_agent.respond("knowledge-demo", "What is The A Way?"))
print("\nSCENARIO 2 - Catalog course without fabricated availability")
print(demo_agent.respond("course-demo", "Is EECE 210 open and who teaches it?"))

In [ ]:
print("SCENARIO 3 - Multi-turn lead collection and real tool persistence")
for turn in ("I want to join CMPS 203.", "My name is Sam Student.", "sam@example.com"):
    print("Student:", turn)
    print("Agent:  ", demo_agent.respond("lead-demo", turn))

In [ ]:
class FeedbackDemoCompletions:
    def __init__(self):
        self.requests = []

    def create(self, **request):
        self.requests.append(request)
        if len(self.requests) == 1:
            message = SimpleNamespace(
                content=None,
                tool_calls=[SimpleNamespace(
                    id="demo-feedback-call",
                    type="function",
                    function=SimpleNamespace(
                        name="record_feedback",
                        arguments=json.dumps({"question": "I have a complaint about tutoring."}),
                    ),
                )],
            )
        else:
            message = SimpleNamespace(
                content="Your feedback was recorded for follow-up; this does not mean it is resolved.",
                tool_calls=[],
            )
        return SimpleNamespace(choices=[SimpleNamespace(message=message)])

demo_provider = SimpleNamespace(chat=SimpleNamespace(completions=FeedbackDemoCompletions()))
demo_settings = Settings("demo-key", "scripted-demo-model", Path(demo_storage.name), 12)
feedback_agent = StudentConcierge(
    client=OpenAIStudentConcierge(settings=demo_settings, client=demo_provider),
    business_knowledge=BUSINESS_KNOWLEDGE,
)
print("SCENARIO 4 - Feedback routed through a genuine model tool-call cycle")
print(feedback_agent.respond("feedback-demo", "I have a complaint about tutoring."))
print("Tool rounds observed:", len(demo_provider.chat.completions.requests))

In [ ]:
print("SCENARIO 5 - Follow-up proves short-term history is retained")
print(demo_agent.respond("memory-demo", "What is The A Way?"))
print(demo_agent.respond("memory-demo", "What did I just ask?"))
print("Stored messages:", len(demo_agent.get_history("memory-demo")))

print("\nSCENARIO 6 - Out-of-scope registration and payment boundary")
print(demo_agent.respond("boundary-demo", "Register me now and take my payment."))

## 7. Gradio ChatInterface

The interface uses a separate bounded conversation per browser session. Uncomment the final line to launch it locally or in Colab. General questions and model-driven feedback require `OPENAI_API_KEY`; deterministic lead slot collection remains local.

In [ ]:
from src.gradio_app import create_interface

if ORIGINAL_DATA_DIR_ENV is None:
    os.environ.pop("THE_A_WAY_DATA_DIR", None)
else:
    os.environ["THE_A_WAY_DATA_DIR"] = ORIGINAL_DATA_DIR_ENV
interface = create_interface()
print("Gradio interface created successfully.")
# interface.launch()

## 8. Run notes

For the live demo, copy `.env.example` to `.env`, set `OPENAI_API_KEY`, restart the kernel, run all cells, and uncomment `interface.launch()`. Never commit `.env`. The complete deterministic audit is `python -m pytest -q`; live integration checks are opt-in as documented in the README.